# 🌧️ ICON-CH1 Rain Map — Leaflet Viewer

Fetches ICON-CH1-EPS precipitation data, saves to `.nc`, and renders an interactive Leaflet map with OSM tiles and labels on top.

## 📥 Step 1 — Fetch + Regrid all lead times (0–33h)

In [ ]:
import xarray as xr
from meteodatalab import ogd_api
from meteodatalab.operators import regrid
from rasterio.crs import CRS
from datetime import timedelta
from earthkit.data import config
import numpy as np

config.set('cache-policy', 'temporary')

# Grid definition (~1km over ICON-CH1 domain)
extent = (-0.817, 18.183, 41.183, 51.183)
nx, ny = 429, 295
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)

das = []
for h in range(0, 34):
    req = ogd_api.Request(
        collection='ogd-forecasting-icon-ch1',
        variable='TOT_PREC',
        reference_datetime='latest',
        perturbed=True,
        horizon=timedelta(hours=h),
    )
    da_h = ogd_api.get_from_ogd(req)
    da_h_geo = regrid.iconremap(da_h, destination)
    das.append(da_h_geo)
    print(f'✓ +{h:02d}h')

da_all = xr.concat(das, dim='lead_time')
da_all = da_all.assign_coords(lead_time=[timedelta(hours=h) for h in range(34)])
print('\nConcatenated shape:', da_all.shape)

## 💾 Step 2 — Clean attrs + compute hourly rain + save to .nc

In [ ]:
def clean_attrs(attrs):
    valid_types = (str, int, float, bytes, list, tuple)
    return {k: v for k, v in attrs.items() if isinstance(v, valid_types)}

da_all.attrs = clean_attrs(da_all.attrs)
for coord in da_all.coords:
    da_all[coord].attrs = clean_attrs(da_all[coord].attrs)

# Ensemble mean hourly rain (diff of cumulative TOT_PREC)
mean_precip = da_all.mean('eps').squeeze('ref_time')       # (lead_time, y, x)
hourly_rain = mean_precip.diff('lead_time')                # (lead_time=33, y, x)
hourly_rain = hourly_rain.drop_vars('valid_time', errors='ignore')
hourly_rain.values = np.where(hourly_rain.values < 0.01, 0.0, np.round(hourly_rain.values, 2))
hourly_rain.attrs = {'long_name': 'Hourly precipitation (ensemble mean)', 'units': 'mm/m2'}

ds = xr.Dataset({
    'TOT_PREC': da_all,
    'hourly_rain': hourly_rain,
})
ds.to_netcdf('icon_ch1_TOT_PREC_all_lead_times.nc')
print('Saved!')

## 🔍 Step 3 — Query a location by lat/lon

In [ ]:
import xarray as xr
import numpy as np

ds = xr.open_dataset('icon_ch1_TOT_PREC_all_lead_times.nc')
da_all = ds['TOT_PREC']
hourly_rain = ds['hourly_rain']

def sel_latlon(da, lat, lon):
    """Select nearest grid cell to a lat/lon point."""
    dist = np.sqrt((da.lat - lat)**2 + (da.lon - lon)**2)
    iy, ix = np.unravel_index(dist.values.argmin(), dist.shape)
    return da.isel(y=iy, x=ix)

# --- Example: Zürich ---
rain_zurich = sel_latlon(hourly_rain, lat=47.37, lon=8.54)
print('Zürich — hourly rain forecast (mm/m²):')
hourly_vals = rain_zurich.values
for h, val in enumerate(hourly_vals, start=1):
    if val > 0.0:
        print(f'  +{h:02d}h → {val:.2f} mm/m²')

# --- Probability at a specific hour ---
tot_zurich = sel_latlon(da_all, lat=47.37, lon=8.54)
h = 18  # hour of interest
rain_h_members = tot_zurich.isel(lead_time=h) - tot_zurich.isel(lead_time=h-1)
prob = float((rain_h_members > 0.1).mean('eps') * 100)
print(f'\nZürich +{h}h probability of rain >0.1mm: {prob:.0f}%')

## 🗺️ Step 4 — Render interactive Leaflet map (labels on top)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import base64, io, xarray as xr

ds = xr.open_dataset('icon_ch1_TOT_PREC_all_lead_times.nc')
hourly_rain = ds['hourly_rain']
max_rain = hourly_rain.max(dim='lead_time').values

# Distance mask — 100km around Switzerland
CH_LAT, CH_LON = 46.8, 8.2
lons = np.linspace(-0.817, 18.183, 429)
lats = np.linspace(41.183, 51.183, 295)
LON, LAT = np.meshgrid(lons, lats)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

within_mask = haversine(LAT, LON, CH_LAT, CH_LON) <= 350

# Colorize
max_val = np.nanmax(max_rain)
norm = mcolors.Normalize(vmin=0, vmax=max_val)
cmap = plt.cm.YlOrRd
max_rain_masked = np.where(within_mask, max_rain, np.nan)
rgba = cmap(norm(np.nan_to_num(max_rain_masked)))
rgba[..., 3] = np.where(within_mask & (max_rain_masked >= 0.01), 1.0, 0)
rgba_flipped = np.flipud(rgba)

# Encode PNG as base64 data URI
buf = io.BytesIO()
plt.imsave(buf, rgba_flipped, format='png')
buf.seek(0)
data_uri = 'data:image/png;base64,' + base64.b64encode(buf.read()).decode('utf-8')

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang=\"en\">
<head>
<meta charset=\"UTF-8\">
<meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\">
<title>ICON-CH1 Rain Map</title>
<link rel=\"stylesheet\" href=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.css\"/>
<script src=\"https://unpkg.com/leaflet@1.9.4/dist/leaflet.js\"></script>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { font-family:system-ui,sans-serif; background:#1a1a1a; color:#eee; }
  #map { width:100vw; height:100vh; }
  #controls {
    position:absolute; top:12px; left:50%; transform:translateX(-50%);
    z-index:1000; background:rgba(20,20,20,0.85); backdrop-filter:blur(8px);
    border-radius:10px; padding:10px 18px; display:flex; align-items:center; gap:16px;
    border:1px solid rgba(255,255,255,0.1);
  }
  #controls label { font-size:13px; color:#aaa; }
  #opacity-slider { width:120px; accent-color:#4f98a3; }
  #opacity-val { font-size:13px; color:#4f98a3; min-width:30px; }
  #legend {
    position:absolute; bottom:30px; right:12px; z-index:1000;
    background:rgba(20,20,20,0.85); backdrop-filter:blur(8px);
    border-radius:10px; padding:12px 16px;
    border:1px solid rgba(255,255,255,0.1); min-width:160px;
  }
  #legend h4 { font-size:11px; color:#aaa; margin-bottom:8px; text-transform:uppercase; letter-spacing:0.05em; }
  .legend-bar {
    height:12px; border-radius:4px; margin-bottom:4px;
    background:linear-gradient(to right,#ffffb2,#fed976,#feb24c,#fd8d3c,#f03b20,#bd0026);
  }
  .legend-labels { display:flex; justify-content:space-between; font-size:11px; color:#888; }
  #info {
    position:absolute; bottom:30px; left:12px; z-index:1000;
    background:rgba(20,20,20,0.85); backdrop-filter:blur(8px);
    border-radius:10px; padding:10px 14px;
    border:1px solid rgba(255,255,255,0.1); font-size:12px; color:#aaa;
  }
  #info strong { color:#eee; }
</style>
</head>
<body>
<div id=\"map\"></div>
<div id=\"controls\">
  <label>Opacity</label>
  <input type=\"range\" id=\"opacity-slider\" min=\"0\" max=\"100\" value=\"65\">
  <span id=\"opacity-val\">65%</span>
</div>
<div id=\"legend\">
  <h4>Max rain – next 33h</h4>
  <div class=\"legend-bar\"></div>
  <div class=\"legend-labels\"><span>0 mm</span><span id=\"legend-max\"></span></div>
</div>
<div id=\"info\">
  <strong>ICON-CH1-EPS</strong><br>
  Max hourly precipitation<br>over next 33 forecast hours
</div>
<script>
const map = L.map('map').setView([46.5, 8.5], 7);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png', {
  attribution:'© OpenStreetMap contributors © CARTO',
  subdomains:'abcd', maxZoom:19
}).addTo(map);
const rainOverlay = L.imageOverlay('__DATA_URI__',
  [[41.183,-0.817],[51.183,18.183]], {opacity:0.65,interactive:false}
).addTo(map);
map.createPane('labelsPane');
map.getPane('labelsPane').style.zIndex = 650;
map.getPane('labelsPane').style.pointerEvents = 'none';
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png', {
  subdomains:'abcd', maxZoom:19, pane:'labelsPane'
}).addTo(map);
document.getElementById('opacity-slider').addEventListener('input', function(){
  rainOverlay.setOpacity(this.value/100);
  document.getElementById('opacity-val').textContent = this.value+'%';
});
document.getElementById('legend-max').textContent = '__MAX_VAL__ mm';
</script>
</body>
</html>"""

html = HTML_TEMPLATE.replace('__DATA_URI__', data_uri)
html = html.replace('__MAX_VAL__', str(round(float(max_val), 2)))

output_path = '/home/ignaz/rain_leaflet.html'
with open(output_path, 'w') as f:
    f.write(html)

print(f'Saved! Open in browser: file://{output_path}')